# 03 — Transformer Comparison: pretrained DistilBERT (SST-2)

`distilbert-base-uncased-finetuned-sst-2-english` ships already trained for sentiment — no fine-tuning needed, so it runs on a normal laptop CPU.

Scoring 3,000 random test reviews is done by `src/compare_transformer.py`; here we pull both models' metrics from MLflow and compare.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # make `src` importable

In [2]:
import mlflow, pathlib
mlflow.set_tracking_uri(f"sqlite:///{pathlib.Path.cwd().parent / 'mlflow.db'}")
rows = []
for exp_name in ['review-radar: baseline', 'review-radar: transformer']:
    exp = mlflow.get_experiment_by_name(exp_name)
    runs = mlflow.search_runs([exp.experiment_id], order_by=['start_time DESC'], max_results=1)
    if len(runs):
        r = runs.iloc[0]
        rows.append({'model': exp_name.split(': ')[1],
                     'f1_negative': r['metrics.f1_negative'],
                     'recall_negative': r['metrics.recall_negative'],
                     'precision_negative': r['metrics.precision_negative']})
import pandas as pd
pd.DataFrame(rows).round(4)

,model,f1_negative,recall_negative,precision_negative
0,baseline,0.8998,0.8961,0.9036
1,transformer,0.8984,0.9125,0.8847


## Takeaway

Both models clear the 0.85 gate. The transformer is a bit stronger, but the baseline is ~100× cheaper to serve and fully explainable — so the **baseline goes to production first**, and the transformer is the upgrade path if the extra points of F1 ever matter more than cost.